In [1]:
import polars as pl

train = pl.read_csv("./data/train.csv")
test = pl.read_csv("./data/test.csv")
submission = pl.read_csv("./data/submission_sample.csv")

In [2]:
train = train.with_columns(
    pl.col('time').str.to_datetime(format="%Y-%m-%d %H:%M:%S%z").alias('time')
)
test = test.with_columns(
    pl.col('time').str.to_datetime(format="%Y-%m-%d %H:%M:%S%z").alias('time')
)

In [3]:
train.shape, test.shape, submission.shape

((19236948, 33), (2838240, 32), (2838240, 4))

In [ ]:
EXCLUDE_COLS = ['time', 'pv_id', 'type', 'energy', 'nins']
features = [col for col in train.columns if col not in EXCLUDE_COLS]

expressions = [
    pl.col(c).fill_null(strategy='backward').over('pv_id').alias(c)
    for c in features
]

train = train.with_columns(expressions)
test = test.with_columns(expressions)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Polars로 빠른 섞기
train_shuffled = train.sample(fraction=1.0, seed=42, shuffle=True)
train_size = int(len(train_shuffled) * 0.8)

train_split = train_shuffled[:train_size]
valid_split = train_shuffled[train_size:]

# 데이터 준비
trainX = train_split.select(features).to_numpy().astype('float32')
trainy = train_split.select('nins').to_numpy().ravel().astype('float32')
validX = valid_split.select(features).to_numpy().astype('float32')
validy = valid_split.select('nins').to_numpy().ravel().astype('float32')

# 정규화
scaler = StandardScaler()
trainX = scaler.fit_transform(trainX)
validX = scaler.transform(validX)

In [15]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# PyTorch 텐서 변환
train_dataset = TensorDataset(
    torch.from_numpy(trainX),
    torch.from_numpy(trainy)
)
valid_dataset = TensorDataset(
    torch.from_numpy(validX),
    torch.from_numpy(validy)
)

train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=4096)

# 모델 정의
class RegressionNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# 학습
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = RegressionNN(len(features)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.L1Loss()  # MAE

best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in (pbar := tqdm(range(200), desc="Training")):
    # Train
    model.train()
    train_loss = 0
    
    for i, (X_batch, y_batch) in enumerate(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch).squeeze()
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
        # 외부 pbar 업데이트 (배치마다)
        pbar.set_postfix({
            'phase': 'train',
            'batch': f'{i+1}/{len(train_loader)}',
            'loss': f'{loss.item():.4f}'
        })
    
    # Validation
    model.eval()
    valid_loss = 0
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(valid_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch).squeeze()
            loss = criterion(pred, y_batch)
            valid_loss += loss.item()
            
            # 외부 pbar 업데이트 (배치마다)
            pbar.set_postfix({
                'phase': 'valid',
                'batch': f'{i+1}/{len(valid_loader)}',
                'loss': f'{loss.item():.4f}'
            })
    
    train_loss /= len(train_loader)
    valid_loss /= len(valid_loader)
    
    # Epoch 종료 후 최종 업데이트
    pbar.set_postfix({
        'train_loss': f'{train_loss:.4f}',
        'valid_loss': f'{valid_loss:.4f}',
        'best': f'{best_loss:.4f}'
    })
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")
    
    # Early Stopping
    if valid_loss < best_loss:
        best_loss = valid_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

# 최고 모델 로드
model.load_state_dict(torch.load('best_model.pt'))

# 예측 (test 데이터도 정규화!)
test_X = scaler.transform(test.select(features).to_numpy().astype('float32'))
test_X = torch.from_numpy(test_X).to(device)

model.eval()
with torch.no_grad():
    pred = model(test_X).cpu().numpy().ravel()

pred[pred < 0] = 0

submission = submission.with_columns(
    pl.Series('nins', pred)
)
submission

Training:   0%|          | 1/200 [02:21<7:50:21, 141.82s/it, train_loss=197.6327, valid_loss=197.6395, best=inf]

Epoch   0 | Train Loss: 197.6327 | Valid Loss: 197.6395


Training:   2%|▏         | 4/200 [10:37<8:40:13, 159.25s/it, phase=train, batch=472/3758, loss=201.3853]             


KeyboardInterrupt: 